In [ ]:
!nvidia-smi

Thu Mar 26 02:53:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers datasets peft bitsandbytes pymatgen pandas numpy matplotlib tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 16.1 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43

In [ ]:
import torch, gc

# Delete old model and free CUDA memory
#del model
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

CUDA available: True
Device: cuda


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

#
CSV_PATH = "zif8_synthesis_data.csv"
df = pd.read_csv(CSV_PATH)
print(df.head())



  Solvent_1 Solvent_2  Ratio  Temperature  Duration Quality
0      MeOH       H2O    2.0           60       3.0    High
1      MeOH       H2O    0.5           60       3.0    High
2      MeOH      MeOH    1.0          100      12.0    High
3      MeOH       H2O    3.0           60       3.0    High
4      MeOH       H2O    4.0           60       3.0    High


In [ ]:
X = df[["Solvent_1",
        "Solvent_2",
        "Ratio",
        "Temperature",
        "Duration"]]

y = df["Quality"]
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (68, 5)
y shape: (68,)


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model


# ==============================
# CONFIG CHUNG
# ==============================
PROJECT_ROOT = Path(".")
PROCESSED_DIR = PROJECT_ROOT / "processed_zif8"
PROCESSED_DIR.mkdir(exist_ok=True)

model_name = "Qwen/Qwen2-1.5B-Instruct"
MAX_LEN = 512

from sklearn.model_selection import StratifiedKFold
cv = 10
RANDOM_STATE = 1
skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)


# ==============================
# BUILD PROMPT
# ==============================
def build_prompt(row):
    prompt = (
        "Instruction: Predict synthesis quality from synthesis conditions.\n\n"
        "Input:\n"
        f"Solvent_1 = {row['Solvent_1']}\n"
        f"Solvent_2 = {row['Solvent_2']}\n"
        f"Ratio = {row['Ratio']}\n"
        f"Temperature = {row['Temperature']}\n"
        f"Duration = {row['Duration']}\n\n"
        "Output:"
    )
    return pd.Series({
        "prompt": prompt,
        "response": str(row["Quality"])
    })

prompt_df = df.apply(build_prompt, axis=1)


# ==============================
# SAVE JSONL
# ==============================
def save_jsonl(samples, path):
    with path.open("w", encoding="utf-8") as f:
        for s in samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")


# ==============================
# TOKENIZER
# ==============================
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ==============================
# TOKENIZE FUNCTION
# ==============================
def tokenize_example(example):
    prompt = example["prompt"]
    response = example["response"]

    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

    prompt_only = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    prompt_tokens = tokenizer(
        prompt_only,
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

    labels = tokenized["input_ids"].copy()
    prompt_len = len(prompt_tokens["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len
    tokenized["labels"] = labels

    return tokenized


# ==============================
# DATA COLLATOR
# ==============================
@dataclass
class DataCollatorForCausalLM:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        labels = [f["labels"] for f in features]

        features_no_labels = []
        for f in features:
            f_copy = {k: v for k, v in f.items() if k != "labels"}
            features_no_labels.append(f_copy)

        batch = DataCollatorWithPadding(
            tokenizer=self.tokenizer,
            padding=True,
            return_tensors="pt"
        )(features_no_labels)

        max_len = batch["input_ids"].shape[1]

        padded_labels = []
        for l in labels:
            l = l[:max_len]
            pad_len = max_len - len(l)
            padded_labels.append(l + [-100] * pad_len)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

data_collator = DataCollatorForCausalLM(tokenizer)


# ==============================
# PREDICT FUNCTION
# ==============================
def predict_quality(model, tokenizer, solvent_1, solvent_2, ratio, temperature, duration):
    prompt_str = (
        "Instruction: Predict synthesis quality from synthesis conditions.\n\n"
        "Input:\n"
        f"Solvent_1 = {solvent_1}\n"
        f"Solvent_2 = {solvent_2}\n"
        f"Ratio = {ratio}\n"
        f"Temperature = {temperature}\n"
        f"Duration = {duration}\n\n"
        "Output:"
    )

    input_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_str}],
        tokenize=False,
        add_generation_prompt=True
    )

    device = next(model.parameters()).device

    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=20,
            do_sample=True,
            top_p=0.9,
            temperature=0.5,
            num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = outputs[0][len(input_ids[0]):]
    predicted_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    predicted_response = predicted_response.split()[0] if len(predicted_response.split()) > 0 else predicted_response

    if predicted_response in ["High", "Low"]:
        return predicted_response
    else:
        return None


# ==============================
# LƯU METRICS QUA 10 LẦN CHẠY
# ==============================
all_accuracies = []
all_precisions = []
all_recalls = []
all_f1s = []
all_conf_mats = []

results_summary = []


# ==============================
# LOOP 10 RANDOM_STATE
# ==============================
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print("=" * 80)
    print(f"RUN fold = {fold}/{cv}")

    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_test  = y.iloc[test_idx].copy()

    print(f"Train size = {len(train_idx)}")
    print(f"Test size  = {len(test_idx)}")

    train_samples = prompt_df.iloc[train_idx].to_dict(orient='records')
    test_samples  = prompt_df.iloc[test_idx].to_dict(orient='records')

    train_path = PROCESSED_DIR / f"train_fold{fold}.jsonl"
    test_path  = PROCESSED_DIR / f"test_fold{fold}.jsonl"

    save_jsonl(train_samples, train_path)
    save_jsonl(test_samples, test_path)


    raw_datasets = load_dataset(
        "json",
        data_files={
            "train": str(train_path),
            "test": str(test_path),
        }
    )

    tokenized_datasets = raw_datasets.map(
        tokenize_example,
        remove_columns=raw_datasets["train"].column_names
    )

    # ------------------------------
    # 3. Load model + QLoRA
    # ------------------------------
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    base_model.config.use_cache = False

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ],
    )

    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

    # ------------------------------
    # 4. Training args
    # ------------------------------
    output_dir = str(PROJECT_ROOT / f"Qwen{fold}")

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=30,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        data_collator=data_collator,
    )

    trainer.train()

    # ------------------------------
    # 5. Prepare test_df
    # ------------------------------
    test_df = X_test.copy()
    test_df["Quality"] = y_test

    # ------------------------------
    # 6. Predict
    # ------------------------------
    y_true = []
    y_pred = []

    for _, r in test_df.iterrows():
        pred = predict_quality(
            trainer.model,
            tokenizer,
            r["Solvent_1"],
            r["Solvent_2"],
            r["Ratio"],
            r["Temperature"],
            r["Duration"]
        )

        if pred is None:
            continue

        y_true.append(r["Quality"])
        y_pred.append(pred)

    # ------------------------------
    # 7. Evaluate
    # ------------------------------
    if len(y_true) == 0:
        print("Không có prediction hợp lệ ở vòng này.")
        continue

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=["High", "Low"])

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(cm)

    all_accuracies.append(acc)
    all_precisions.append(prec)
    all_recalls.append(rec)
    all_f1s.append(f1)
    all_conf_mats.append(cm)

    results_summary.append({
        "random_state": fold,
        "accuracy": acc,
        "precision_weighted": prec,
        "recall_weighted": rec,
        "f1_weighted": f1,
        "n_eval_samples": len(y_true),
    })

    # Giải phóng bớt VRAM
    del trainer
    del model
    del base_model
    torch.cuda.empty_cache()


# ==============================
# 8. IN KẾT QUẢ TỪNG LẦN CHẠY
# ==============================
print("\n" + "=" * 80)
print("KẾT QUẢ TỪNG LẦN CHẠY")
results_df = pd.DataFrame(results_summary)
print(results_df)


# ==============================
# 9. IN KẾT QUẢ TRUNG BÌNH
# ==============================
if len(all_accuracies) > 0:
    print("\n" + "=" * 80)
    print("KẾT QUẢ TRUNG BÌNH CỦA 10 VÒNG LẶP")
    print(f"Average Accuracy : {np.mean(all_accuracies):.4f}")
    print(f"Average Precision: {np.mean(all_precisions):.4f}")
    print(f"Average Recall   : {np.mean(all_recalls):.4f}")
    print(f"Average F1-score : {np.mean(all_f1s):.4f}")

    avg_cm = np.mean(np.array(all_conf_mats), axis=0)
    sum_cm = np.sum(np.array(all_conf_mats), axis=0)

    print("\nAverage Confusion Matrix:")
    print(avg_cm)

    print("\nSummed Confusion Matrix:")
    print(sum_cm)
else:
    print("Không có vòng chạy nào cho ra prediction hợp lệ.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


Step,Training Loss
10,0.564929
20,0.173453
30,0.117616
40,0.065026
50,0.051674
60,0.040572
70,0.028576
80,0.039266
90,0.016151
100,0.018711


Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-score : 1.0000

Classification Report:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00         5
         Low       1.00      1.00      1.00         1

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6


Confusion Matrix:
[[5 0]
 [0 1]]

KẾT QUẢ TỪNG LẦN CHẠY
   random_state  accuracy  precision_weighted  recall_weighted  f1_weighted  \
0             1  0.857143            0.928571         0.857143     0.874459   
1             2  1.000000            1.000000         1.000000     1.000000   
2             3  1.000000            1.000000         1.000000     1.000000   
3             4  0.857143            0.880952         0.857143     0.839827   
4             5  1.000000            1.000000         1.000000     1.000000   
5             6  0.714286            0.510204 